# Comparing Metaflow autotermination, checkpointing, and `spin` for debugging long flows

## Purpose

When a step in a Metaflow flow runs for hours — a hyperparameter sweep, distributed training, or a large data transformation — debugging becomes expensive. Re-running the entire flow from scratch on every iteration wastes compute and time. Metaflow provides three complementary mechanisms for working with long-running steps:

- **Autotermination** via `@timeout`: forcefully interrupts a step that exceeds a time budget, preventing wasted compute on stuck or runaway tasks.
- **Checkpointing** via `@checkpoint` (with `@retry`): periodically persists in-task progress so that a restarted step resumes from the last checkpoint instead of from scratch.
- **`spin`**: runs a single step transiently against the prior run's inputs without recording metadata, making it ideal for rapid iteration on a single step's logic.

This notebook defines a shared `FlowSpec` that simulates a long training loop, then shows how each strategy applies to the same code path.

## When to use

| Strategy | When to use it | When to avoid |
|---|---|---|
| `@timeout` (autotermination) | A step may hang indefinitely (deadlocks, resource starvation); you need a hard ceiling on runtime | The step genuinely needs more time and has no natural checkpoint boundaries |
| `@checkpoint` + `@retry` | Training or processing that can be split into resumable units; failures are transient (spot interruptions, network glitches) | Steps with non-deterministic state that cannot be safely restored |
| `spin` | Iterating on a single step's logic; unit testing step code with real artifacts from a prior run; CI smoke tests | Producing artifacts that must be stored for downstream steps or audit |

## Prerequisites

- Metaflow installed with a backend configured (`--with batch` or `--with kubernetes`)
- The `metaflow-checkpoint` extension installed for in-task checkpointing
- At least one prior successful run of the flow (required for `spin`, which pulls inputs from the most recent run)
- Python 3.9+

In [ ]:
from metaflow import FlowSpec, step, current, Parameter, timeout, retry, checkpoint


class DebugLongFlow(FlowSpec):
    """
    Simulated long-running training flow.
    All three debugging strategies apply to the same core logic.
    """

    iterations = Parameter(
        "iterations",
        help="Total training iterations to simulate",
        default=500,
    )

    @step
    def start(self):
        """Simulate loading a dataset."""
        self.dataset = {"features": list(range(100)), "n_features": 100}
        print(f"Loaded dataset with {self.dataset['n_features']} features.")
        self.next(self.train)

    @timeout(minutes=5)
    @retry(times=3, minutes_between_retries=1)
    @checkpoint
    @step
    def train(self):
        """
        Long training loop with:
        - @timeout: hard ceiling of 5 minutes per attempt
        - @retry: up to 3 attempts with 1-minute backoff
        - @checkpoint: saves progress every 100 iterations
        """
        from time import sleep
        import os

        cp_dir = current.checkpoint.directory
        progress_file = os.path.join(cp_dir, "train_progress.txt")

        start_iter = 0
        if current.checkpoint.is_loaded:
            if os.path.exists(progress_file):
                with open(progress_file) as f:
                    start_iter = int(f.read().strip())
                print(f"Resuming from iteration {start_iter}")

        for i in range(start_iter, self.iterations):
            sleep(0.01)  # simulate per-iteration compute

            if i > start_iter and i % 100 == 0:
                with open(progress_file, "w") as f:
                    f.write(str(i))
                current.checkpoint.save()
                print(f"Checkpoint saved at iteration {i}")

        self.model_state = {"trained": True, "iterations": self.iterations}
        self.next(self.end)

    @step
    def end(self):
        print(f"Training complete: {self.model_state['iterations']} iterations.")


if __name__ == "__main__":
    DebugLongFlow()

### 1. Autotermination with `@timeout`

The `@timeout` decorator sets a hard ceiling on step execution time. When the step exceeds the limit, Metaflow interrupts it and treats the timeout as an exception. Combined with `@retry`, the step is automatically retried. Without `@checkpoint`, the retry starts over from the beginning; with `@checkpoint`, it resumes from the last saved progress.

```python
from metaflow import timeout

# @timeout accepts seconds, minutes, and hours (cumulative)
@timeout(minutes=5)
@step
def train(self):
    ...
```

Run with timeout enforcement enabled:

```bash
# The timeout applies on every backend (local, @batch, @kubernetes)
python debug_long_flow.py run --with batch
```

### 2. Checkpointing with `@checkpoint` and `@retry`

The `@checkpoint` decorator (from the `metaflow-checkpoint` extension) designates a step for in-task checkpointing. A local staging directory is available at `current.checkpoint.directory`. Files written there are persisted to the Metaflow datastore when `current.checkpoint.save()` is called. On a retried attempt, Metaflow loads the latest checkpoint automatically and sets `current.checkpoint.is_loaded` to `True`.

Key behaviors:
- Checkpoints are scoped per-task, so `@foreach` branches each receive independent checkpoints.
- Checkpoint files persist to the datastore, surviving container restarts in AWS Batch or Kubernetes.
- Pairing `@checkpoint` with `@retry` means a failed step restarts from the last checkpoint rather than from scratch.

### 3. Transient debugging with `spin`

The `spin` command runs a single step (or subset of steps) as a transient run. Unlike a normal run, `spin` does not persist artifacts to the metadata service — results are stored in a local `.metaflow_spin/` directory. This makes `spin` ideal for rapid iteration on a single step's logic without polluting the metadata store.

`spin` uses the exact same input artifacts as the most recent successful run by default. Individual artifacts can be overridden with an `--artifacts-module` flag.

```python
from metaflow import Runner

# Run only the "train" step transiently, reusing the latest run's inputs.
# persist=True writes artifacts to .metaflow_spin/ for later inspection.
with Runner("debug_long_flow.py").spin("train", persist=True) as spinning:
    task = spinning.task
    print(f"Spin task ID: {task.get('task')}")
```

Equivalent CLI usage:

```bash
# Transient run of just the train step
python debug_long_flow.py spin train

# Persist outputs for inspection
python debug_long_flow.py spin train --persist

# Override specific input artifacts
python debug_long_flow.py spin train --artifacts-module my_artifacts.py
```

### Combining all three

The three strategies compose naturally. A typical pattern for a long training step:

1. `@timeout(minutes=30)` — kill the step if it hangs
2. `@retry(times=5, minutes_between_retries=2)` — retry on failure
3. `@checkpoint` — save progress every N iterations so retries resume, not restart
4. During development, use `spin train` instead of `run train` for faster iteration

With this combination, a spot interruption at iteration 400 of 1000 doesn't waste 40% of compute — the retry picks up from the checkpoint at iteration 300.

## Verify

1. **Timeout enforcement**: Add `@timeout(seconds=5)` to a step containing `time.sleep(10)`. Run locally and confirm the step is interrupted after 5 seconds with a timeout exception.

2. **Checkpoint recovery**: Run the flow with a forced failure (raise an exception mid-loop), then re-run with `--resume`. Confirm the resumed step prints `Resuming from iteration N` where N matches the last checkpoint.

3. **Spin execution**: Run `python debug_long_flow.py spin train` and confirm a `.metaflow_spin/` directory is created without an entry appearing in `python debug_long_flow.py list`.

## Common errors

- **`ImportError` for `checkpoint`**: The `@checkpoint` decorator comes from the `metaflow-checkpoint` extension, not core Metaflow. Install it separately before importing: `from metaflow import checkpoint`.
- **`spin` finds no prior run**: `spin` requires a prior successful run of the same flow. If no run exists, `spin` exits with an error. Run `python debug_long_flow.py run` once before attempting `spin`.
- **Timeout cancels retry progress**: If `@timeout` kills the step during a non-checkpointed section, `@retry` restarts from the beginning. Always pair `@timeout` with `@checkpoint` for expensive steps.
- **Checkpoint directory collision in foreach**: Each foreach split gets its own checkpoint namespace automatically. Manually writing to a hardcoded path instead of `current.checkpoint.directory` can cause splits to overwrite each other's progress.
- **`minutes_between_retries` too short**: Setting a backoff shorter than the time to provision a new container means the retry starts before infrastructure is ready. Use at least 30 seconds for AWS Batch steps.